# Prefix-Conditioned Rollout Exploration (PCRE)

Standard GRPO penalises **every token** of an incorrect rollout, even perfectly reasonable prefixes.
PCRE fixes this: take past rollouts, truncate to a prefix, and make the model complete from there.
Reward is applied only to the generated suffix — the prefix tokens receive no gradient.

**This notebook has 3 cells:**
1. Load model + dataset
2. GRPO data: prompt format, reward, loss mask
3. Prefix-conditioned data: buffer, prefix prompt, reward on suffix, loss mask

In [ ]:
## Cell 1 — Load model, tokenizer, dataset

import os, sys, random, torch
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

sys.path.insert(0, os.path.abspath('.'))
from src.prefix_rollout import PrefixRolloutBuffer
from src.nb_utils import render_grad_html, generate_rollout, is_correct

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
model.eval()

dataset = load_dataset("openai/gsm8k", "main")

def _extract_gold(text):
    import re
    m = re.search(r"####\s*(.+)", text)
    return m.group(1).strip().replace(",", "") if m else ""

def format_example(ex):
    ex["prompt"]      = [{"role": "user", "content": ex["question"]}]
    ex["gold_answer"] = _extract_gold(ex["answer"])
    return ex

train_dataset = dataset["train"].map(format_example)
print(f"Model : {MODEL_NAME}  |  device: {next(model.parameters()).device}")
print(f"Train : {len(train_dataset)} examples  |  Gold[0]: {train_dataset[0]['gold_answer']}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
## Cell 2 — GRPO: real rollout, reward, loss mask

Q    = train_dataset[0]["question"]
GOLD = train_dataset[0]["gold_answer"]

std_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": Q}],
    tokenize=False, add_generation_prompt=True
)

print(f"Question : {Q}")
print(f"Gold     : {GOLD}")
print("Generating rollout (may take ~10–30s)...")

prompt_str, completion = generate_rollout(model, tokenizer, Q, max_new_tokens=400)
correct = is_correct(completion, GOLD)
reward  = 1.0 if correct else 0.0

print(f"\n--- Completion ---\n{completion}")
print(f"\nCorrect: {correct}  |  Reward: {reward}")

# ── Loss mask visualisation ───────────────────────────────────────────────
std_ids  = tokenizer(std_prompt,  return_tensors="pt")["input_ids"]
comp_ids = tokenizer(completion,  add_special_tokens=False, return_tensors="pt")["input_ids"]

render_grad_html(
    no_grad_text = std_prompt,
    grad_text    = completion,
    n_no_grad    = std_ids.shape[1],
    n_grad       = comp_ids.shape[1],
)

Reward [CORRECT]: 1.0
Reward [WRONG]: 0.0


In [ ]:
## Cell 3 — Prefix-conditioned: fill buffer with real rollouts, prefix sample, loss mask

N_ROLLOUTS = 6   # adjust as needed; each takes ~10–30s

buf = PrefixRolloutBuffer(max_size=200, min_prefix_frac=0.20, max_prefix_frac=0.75)

print(f"Generating {N_ROLLOUTS} rollouts to fill buffer...")
for i in range(N_ROLLOUTS):
    ex   = train_dataset[i]
    q, gold = ex["question"], ex["gold_answer"]
    _, comp  = generate_rollout(model, tokenizer, q, max_new_tokens=300)
    ok   = is_correct(comp, gold)
    r    = 1.0 if ok else 0.0
    buf.add(q, comp, ok, reward=r)
    print(f"  [{i+1}/{N_ROLLOUTS}] correct={ok}  reward={r}  | {comp[:60]!r}...")

print("\nBuffer stats:", buf.get_stats())

# ── Prefix sampling (from correct rollouts only) ──────────────────────────
base_item = dict(train_dataset[0])
random.seed(None)   # live random seed
aug = buf.sample_prefix_augmented_item(base_item, tokenizer, from_correct=True)

if aug is None:
    print("\nNo correct rollouts in buffer yet — try N_ROLLOUTS=10 or from_correct=None")
else:
    prefix_prompt_str = aug["prompt"]
    prefix_ids = tokenizer(prefix_prompt_str, return_tensors="pt")["input_ids"]

    # Mock suffix: what the model would generate after the prefix
    # In real training TRL generates this; here we show the structure
    SUFFIX_DEMO = "(model generates the remaining reasoning + answer here)"

    suffix_ids = tokenizer(SUFFIX_DEMO, add_special_tokens=False, return_tensors="pt")["input_ids"]
    comp_ids_ref = tokenizer(buf._buf[0]["completion"], add_special_tokens=False, return_tensors="pt")["input_ids"]
    n_saved = comp_ids_ref.shape[1] - suffix_ids.shape[1]

    print(f"\nPrefix prompt ends at: {prefix_prompt_str[-80:]!r}")
    print(f"Prefix prompt tokens : {prefix_ids.shape[1]}  (no gradient)")
    print(f"Suffix tokens        : {suffix_ids.shape[1]}  (gradient only)")

    render_grad_html(
        no_grad_text = prefix_prompt_str,
        grad_text    = SUFFIX_DEMO,
        n_no_grad    = prefix_ids.shape[1],
        n_grad       = suffix_ids.shape[1],
        n_saved      = max(0, n_saved),
    )

print(f"\nCorrect in buffer  : {sum(1 for r in buf._buf if r['correct'])}")
print(f"Incorrect in buffer: {sum(1 for r in buf._buf if not r['correct'])}")

Buffer stats: {'buffer_size': 4, 'n_correct': 2, 'n_incorrect': 2, 'mean_reward': 0.5}

Buffer contents:
  correct=True  reward=1.0  | '<think>\nIn April she sold 48 clips.\nIn May half as many: 48/'...
  correct=False  reward=0.0  | '<think>\nIn April 48 clips, in May half: 24.\nTotal: 48 - 24 ='...
  correct=True  reward=1.0  | '<think>\nWeng earns $12/hr.\n50 min = 50/60 hr.\n12×5/6 = 10.\n<'...
  correct=False  reward=0.0  | '<think>\nBetty needs $100, has $50.\nParents give $15, grandpa'...

Reward on suffix: 1.0  (Gold=72)



Correct prefixes  : 2
Incorrect prefixes: 2
Upsample hint     : sorted(buf._buf, key=lambda r: -r['reward']) for rare-correct emphasis
